In [1]:
import json
import re
import os
import pathlib
import collections
from datetime import datetime

# Cấu hình đường dẫn
INPUT_FILE = pathlib.Path('../data/du_lieu_phap_luat.json')
OUTPUT_DIR = pathlib.Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"File đầu vào: {INPUT_FILE.resolve()}")
print(f"Thư mục đầu ra: {OUTPUT_DIR.resolve()}")


File đầu vào: D:\Đồ án tn\legal-ai-datn\backend\data\du_lieu_phap_luat.json
Thư mục đầu ra: D:\Đồ án tn\legal-ai-datn\backend\data\processed


## 1. Chuẩn Hóa Số Thứ Tự & Quy Ước Đặt ID (`node_id`)

Quy ước định danh cho từng cấp cấu trúc:
- `Document`: `{DOC_ID}` (ví dụ: `HIEN_PHAP_2013`, `BLDS_2015`)
- `Part` (Phần): `{DOC_ID}_P{n}` (ví dụ: `BLDS_2015_P1`)
- `Chapter` (Chương): `{DOC_ID}_CH{n}` hoặc `{DOC_ID}_P{p}_CH{n}`
- `Section` (Mục): `{DOC_ID}_CH{ch}_M{n}`
- `SubSection` (Tiểu mục): `{DOC_ID}_CH{ch}_M{m}_TM{n}` (ví dụ: `BLDS_2015_CH13_M1_TM1`)
- `Article` (Điều): `{DOC_ID}_D{n}` (ví dụ: `BLDS_2015_D584`, `HIEN_PHAP_2013_D1`)
- `Clause` (Khoản): `{DOC_ID}_D{dieu}_K{khoan}` (ví dụ: `BLDS_2015_D584_K1`)
- `Point` (Điểm): `{DOC_ID}_D{dieu}_K{khoan}_DIEM_{diem}` (ví dụ: `BLDS_2015_D584_K1_DIEM_A`)

> **Điểm mấu chốt:** SubSection được định danh gắn liền với parent context (`_CH{ch}_M{m}_TM{n}`) để đảm bảo 100% tính duy nhất toàn cục. Điểm dùng tiền tố `DIEM_` để tránh nhầm lẫn với Điều (`D`).


In [2]:
# Bảng ánh xạ số La Mã và từ thứ tự tiếng Việt sang số nguyên
VIETNAMESE_ORDINALS = {
    'nhất': 1, 'thứ nhất': 1, 'hai': 2, 'thứ hai': 2, 'ba': 3, 'thứ ba': 3,
    'bốn': 4, 'thứ tư': 4, 'thứ bốn': 4, 'tư': 4, 'năm': 5, 'thứ năm': 5,
    'sáu': 6, 'thứ sáu': 6, 'bảy': 7, 'thứ bảy': 7, 'tám': 8, 'thứ tám': 8,
    'chín': 9, 'thứ chín': 9, 'mười': 10, 'thứ mười': 10
}

ROMAN_NUMERALS = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9, 'X': 10,
    'XI': 11, 'XII': 12, 'XIII': 13, 'XIV': 14, 'XV': 15, 'XVI': 16, 'XVII': 17, 'XVIII': 18, 'XIX': 19, 'XX': 20,
    'XXI': 21, 'XXII': 22, 'XXIII': 23, 'XXIV': 24, 'XXV': 25, 'XXVI': 26, 'XXVII': 27, 'XXVIII': 28, 'XXIX': 29, 'XXX': 30
}

def normalize_number(num_val):
    """Chuẩn hóa số nguyên, số La Mã hoặc từ thứ tự tiếng Việt về số nguyên."""
    if num_val is None:
        return ''
    s = str(num_val).strip()
    if s.isdigit():
        return int(s)
    s_lower = s.lower()
    if s_lower in VIETNAMESE_ORDINALS:
        return VIETNAMESE_ORDINALS[s_lower]
    s_upper = s.upper()
    if s_upper in ROMAN_NUMERALS:
        return ROMAN_NUMERALS[s_upper]
    return s

LOAI_TO_TYPE = {
    'van_ban': 'Document',
    'phan': 'Part',
    'chuong': 'Chapter',
    'muc': 'Section',
    'tieu_muc': 'SubSection',
    'dieu': 'Article',
    'khoan': 'Clause',
    'diem': 'Point',
}

CHILD_KEYS = ['cac_phan', 'cac_chuong', 'cac_muc', 'cac_tieu_muc', 'cac_dieu', 'cac_khoan', 'cac_diem', 'children']

def get_node_id(doc_id, item, context):
    """Sinh ID chuẩn hóa dựa trên loại node và ngữ cảnh cha con."""
    loai = item.get('loai')
    raw_so = item.get('so')
    norm_so = normalize_number(raw_so)
    
    if loai == 'van_ban':
        return doc_id
    elif loai == 'phan':
        return f"{doc_id}_P{norm_so}"
    elif loai == 'chuong':
        part_so = context.get('part_so')
        part_prefix = f"_P{part_so}" if part_so else ""
        return f"{doc_id}{part_prefix}_CH{norm_so}"
    elif loai == 'muc':
        chuong_so = context.get('chuong_so')
        ch_prefix = f"_CH{chuong_so}" if chuong_so else ""
        return f"{doc_id}{ch_prefix}_M{norm_so}"
    elif loai == 'tieu_muc':
        chuong_so = context.get('chuong_so')
        muc_so = context.get('muc_so')
        ch_prefix = f"_CH{chuong_so}" if chuong_so else ""
        m_prefix = f"_M{muc_so}" if muc_so else ""
        return f"{doc_id}{ch_prefix}{m_prefix}_TM{norm_so}"
    elif loai == 'dieu':
        return f"{doc_id}_D{norm_so}"
    elif loai == 'khoan':
        dieu_so = context.get('dieu_so')
        return f"{doc_id}_D{dieu_so}_K{norm_so}"
    elif loai == 'diem':
        dieu_so = context.get('dieu_so')
        khoan_so = context.get('khoan_so')
        diem_str = str(raw_so).strip().upper()
        if khoan_so:
            return f"{doc_id}_D{dieu_so}_K{khoan_so}_DIEM_{diem_str}"
        else:
            return f"{doc_id}_D{dieu_so}_DIEM_{diem_str}"
    else:
        return f"{doc_id}_{loai.upper()}_{norm_so}"
print("Đã tải thành công các hàm chuẩn hóa ID.")


Đã tải thành công các hàm chuẩn hóa ID.


## 2. Thuật Toán Duyệt Cây Đệ Quy Tổng Quát (Generic Recursive Traversal)

Thuật toán giải quyết triệt để vấn đề đa dạng cấu trúc giữa các văn bản pháp luật:
- **Không phụ thuộc độ sâu cố định:** Tự động phát hiện mọi danh sách con (`cac_phan`, `cac_chuong`, `cac_muc`, `cac_tieu_muc`, `cac_dieu`, `cac_khoan`, `cac_diem`, `children`).
- **Tự động gắn quan hệ cấu trúc:** Luôn tạo `(parent)-[:CONTAINS]->(child)` với `source_type = 'STRUCTURAL'`, `confidence = 1.0`.
- **Bảo tồn ngữ cảnh cha con:** Truyền số thứ tự của Phần, Chương, Điều, Khoản xuống các tầng con để phục vụ sinh ID.
- **Sinh `full_text` cho Article:** Tự động gộp Tiêu đề + Nội dung Điều + toàn bộ các Khoản và Điểm con thành trường `full_text` hoàn chỉnh, tối ưu cho bài toán Embedding/GraphRAG sau này.


In [3]:
def traverse_tree_recursive(item, parent_id, doc_id, context, nodes, contains_relations, seen_node_ids=None):
    """
    Duyệt cây đệ quy tổng quát (Generic Recursive Traversal)
    Tự động thích ứng với mọi độ sâu cấu trúc văn bản:
    - Bất kể văn bản có bao nhiêu tầng, hàm tự động kết nối (parent)-[:CONTAINS]->(child)
    - Tự động truyền ngữ cảnh phân cấp (part_so, chuong_so, dieu_so, khoan_so)
    """
    if seen_node_ids is None:
        seen_node_ids = set()
        
    loai = item.get('loai')
    if not loai:
        return
        
    node_type = LOAI_TO_TYPE.get(loai, loai.capitalize())
    base_node_id = get_node_id(doc_id, item, context)
    
    # Đảm bảo tính duy nhất tuyệt đối (V1)
    if loai == 'van_ban':
        current_node_id = doc_id
    else:
        current_node_id = base_node_id
        if current_node_id in seen_node_ids:
            dup_idx = 2
            while f"{base_node_id}_{dup_idx}" in seen_node_ids:
                dup_idx += 1
            current_node_id = f"{base_node_id}_{dup_idx}"
        seen_node_ids.add(current_node_id)
    
    # Cập nhật ngữ cảnh cho con cháu
    new_context = dict(context)
    norm_so = normalize_number(item.get('so'))
    if loai == 'phan':
        new_context['part_so'] = norm_so
    elif loai == 'chuong':
        new_context['chuong_so'] = norm_so
    elif loai == 'muc':
        new_context['muc_so'] = norm_so
    elif loai == 'dieu':
        new_context['dieu_so'] = norm_so
    elif loai == 'khoan':
        new_context['khoan_so'] = norm_so
        
    # Tạo node (loại trừ gốc van_ban đã được tạo trước)
    if loai != 'van_ban':
        title = item.get('ten') or item.get('tieu_de')
        text = item.get('noi_dung')
        number = item.get('so')
        
        node = {
            'node_id': current_node_id,
            'document_id': doc_id,
            'type': node_type,
            'label': node_type,
            'number': number,
            'title': title,
            'text': text,
            'parent_id': parent_id
        }
        if loai == 'dieu':
            full_parts = []
            if title:
                full_parts.append(title.strip())
            if text:
                full_parts.append(text.strip())
            for k in item.get('cac_khoan', []):
                k_so = k.get('so')
                k_noi_dung = k.get('noi_dung')
                if k_noi_dung:
                    full_parts.append(f'Khoản {k_so}. {k_noi_dung.strip()}')
            node['full_text'] = '\n'.join(full_parts) if full_parts else None
            
        nodes.append(node)
        
        if parent_id:
            contains_relations.append({
                'source_id': parent_id,
                'type': 'CONTAINS',
                'target_id': current_node_id,
                'evidence': f"{parent_id} CONTAINS {current_node_id}",
                'source_type': 'STRUCTURAL',
                'confidence': 1.0
            })
            
    # Duyệt đệ quy tất cả danh sách con
    child_parent_id = current_node_id
    for k, v in item.items():
        if k in CHILD_KEYS or (k.startswith('cac_') and isinstance(v, list)):
            for child in v:
                if isinstance(child, dict) and 'loai' in child:
                    traverse_tree_recursive(child, child_parent_id, doc_id, new_context, nodes, contains_relations, seen_node_ids)
print("Đã tải hàm duyệt cây đệ quy tổng quát.")


Đã tải hàm duyệt cây đệ quy tổng quát.


## 3. Trích Xuất Dẫn Chiếu Pháp Lý (`REFERENCES`) Có Bằng Chứng Ngữ Cảnh

Nguyên tắc trích xuất dẫn chiếu:
* Chỉ tạo quan hệ `REFERENCES` khi có dẫn chiếu tường minh trong văn bản.
* Mọi quan hệ `REFERENCES` bắt buộc phải có `evidence` (trích xuất cả câu/đoạn văn bản chứa từ khóa).
* `source_type = 'EXPLICIT'`, `confidence = 1.0` (độ tin cậy trích xuất cú pháp).
* Tìm target node chính xác nhất (`Điểm` $\rightarrow$ `Khoản` $\rightarrow$ `Điều`).

### Các Bộ Xử Lý Dẫn Chiếu Phức (Pattern Extractors):
1. **Range Điều (Article Range):** `từ Điều X đến Điều Y` $\rightarrow$ Tự động mở rộng (expand) thành toàn bộ các Điều liên tục từ $X$ đến $Y$.
2. **List Điểm cùng Khoản/Điều (Point List):** `điểm X và điểm Y khoản K Điều W` $\rightarrow$ Sinh độc lập các node Điểm tương ứng.
3. **List Khoản cùng Điều (Clause List):** `khoản X và khoản Y Điều Z` $\rightarrow$ Sinh độc lập các node Khoản tương ứng.
4. **List Điều (Article List):** `[các] Điều X, Y và Z` $\rightarrow$ Sinh độc lập các node Điều tương ứng.
5. **Dẫn chiếu nội bộ Điều này (Internal Article References):** `[điểm X] [khoản Y] Điều này` $\rightarrow$ Tự động neo vào Điều hiện tại của node nguồn.
6. **Single Citation (Dẫn chiếu đơn lẻ):** `[Điểm X] [Khoản Y] Điều Z`.


In [ ]:
"""
relation_parser.py
Module trích xuất quan hệ pháp lý.
Hỗ trợ đầy đủ:
1. Range Điều (từ Điều X đến [Điều] Y)
2. List Điểm cùng Khoản/Điều (điểm X và điểm Y khoản K Điều W)
3. List Khoản cùng Điều (khoản X và khoản Y Điều Z, khoản 1, 3 và 4 Điều W)
4. List Điều (các điều X, Y và Z; Điều X, Điều Y)
5. Dẫn chiếu nội bộ Điều này ([điểm X] [khoản Y] Điều này; khoản X và khoản Y Điều này)
6. Single Citation thông thường ([Điểm X] [Khoản Y] Điều Z)
"""

import re
from typing import List, Dict, Any, Set


def get_evidence(text: str, start: int, end: int, window: int = 35) -> str:
    """Trích xuất chuỗi bằng chứng ngữ cảnh bao quanh vị trí khớp Regex."""
    ctx_start = max(0, start - window)
    ctx_end = min(len(text), end + window)
    raw = text[ctx_start:ctx_end].strip()
    return f"...{raw}..." if (ctx_start > 0 or ctx_end < len(text)) else raw


def extract_references(nodes: List[Dict[str, Any]], node_id_set: Set[str]) -> List[Dict[str, Any]]:
    """
    Trích xuất toàn bộ quan hệ dẫn chiếu pháp lý REFERENCES từ nội dung văn bản của các nodes.
    
    Args:
        nodes: Danh sách các node trong đồ thị (phải có node_id, document_id, text).
        node_id_set: Tập hợp tất cả các node_id hợp lệ trong đồ thị để kiểm tra tính tồn tại.
        
    Returns:
        Danh sách các quan hệ REFERENCES chuẩn hóa, có evidence, không trùng lặp và không self-loop.
    """
    # 1. Regex Range Điều: từ Điều X đến [Điều] Y
    range_dieu_rx = re.compile(
        r'từ\s+[Đđ]iều\s+(\d+)\s+đến\s+(?:[Đđ]iều\s+)?(\d+)',
        re.IGNORECASE
    )
    
    # 2. Regex List Điểm cùng Khoản/Điều
    list_diem_rx = re.compile(
        r'(?:[Đđ]iểm\s+([a-zđ])(?:\s*(?:,|và|hoặc)\s*(?:[Đđ]iểm\s+)?([a-zđ]))+)\s+(?:(?:của\s+)?[Kk]hoản\s+(\d+)\s+)?(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )
    
    # 3. Regex List Khoản cùng Điều
    list_khoan_rx = re.compile(
        r'(?:[Kk]hoản\s+(\d+)(?:\s*(?:,|và|hoặc)\s*(?:[Kk]hoản\s+)?(\d+))+)\s+(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )
    
    # 4. Regex List Điều
    list_dieu_rx = re.compile(
        r'(?:(?:các\s+)?[Đđ]iều\s+(\d+)(?:\s*(?:,|và)\s*(?:[Đđ]iều\s+)?(\d+))+)',
        re.IGNORECASE
    )
    
    # 5. Regex Dẫn chiếu nội bộ "Điều này"
    internal_dieu_rx = re.compile(
        r'(?:[Đđ]iểm\s+([a-zđ])(?:\s*(?:,|và|hoặc)\s*(?:[Đđ]iểm\s+)?([a-zđ]))*\s+)?(?:[Kk]hoản\s+(\d+)(?:\s*(?:,|và|hoặc)\s*(?:[Kk]hoản\s+)?(\d+))*\s+)?(?:của\s+)?[Đđ]iều\s+này',
        re.IGNORECASE
    )
    
    # 6. Regex Single Citation thông thường
    single_ref_rx = re.compile(
        r'(?:(?:[Đđ]iểm\s+([a-zđ]))\s+)?(?:(?:[Kk]hoản\s+(\d+))\s+)?(?:của\s+)?[Đđ]iều\s+(\d+)',
        re.IGNORECASE
    )

    all_refs = []

    for node in nodes:
        text = node.get('text')
        if not text:
            continue
        source_id = node['node_id']
        doc_id = node['document_id']
        
        # Trích xuất số Điều hiện tại của node nguồn (nếu có) để phục vụ giải quyết "Điều này"
        cur_dieu_match = re.search(r'_D(\d+)', source_id)
        cur_dieu = cur_dieu_match.group(1) if cur_dieu_match else None
        
        # ----------------------------------------------------
        # Pattern 1: Range Điều (từ Điều X đến Điều Y)
        # ----------------------------------------------------
        for match in range_dieu_rx.finditer(text):
            start_num = int(match.group(1))
            end_num = int(match.group(2))
            if 1 <= start_num < end_num and (end_num - start_num) <= 50:
                ev = get_evidence(text, match.start(), match.end())
                for d in range(start_num, end_num + 1):
                    target_id = f"{doc_id}_D{d}"
                    if target_id in node_id_set and target_id != source_id:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': target_id,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # ----------------------------------------------------
        # Pattern 2: List Điểm cùng Khoản/Điều
        # ----------------------------------------------------
        for match in list_diem_rx.finditer(text):
            full_match_text = match.group(0)
            dieu_num = match.group(4) if match.group(4) else match.group(3)
            khoan_match = re.search(r'[Kk]hoản\s+(\d+)', full_match_text)
            khoan_num = khoan_match.group(1) if khoan_match else None
            
            diem_letters = re.findall(
                r'[Đđ]iểm\s+([a-zđ])|(?:\\s*,\\s*|\\s+và\\s+|\\s+hoặc\\s+)([a-zđ])\\b',
                full_match_text[:full_match_text.lower().find('điều')],
                re.IGNORECASE
            )
            points = []
            for g1, g2 in diem_letters:
                letter = g1 or g2
                if letter and letter.lower() not in ['và', 'của', 'hoặc']:
                    points.append(letter.upper())
                    
            ev = get_evidence(text, match.start(), match.end())
            for p in points:
                candidates = []
                if khoan_num:
                    candidates.append(f"{doc_id}_D{dieu_num}_K{khoan_num}_DIEM_{p}")
                    candidates.append(f"{doc_id}_D{dieu_num}_K{khoan_num}")
                else:
                    candidates.append(f"{doc_id}_D{dieu_num}_DIEM_{p}")
                candidates.append(f"{doc_id}_D{dieu_num}")
                
                target_id = None
                for c in candidates:
                    if c in node_id_set:
                        target_id = c
                        break
                if not target_id:
                    target_id = candidates[0]
                    
                if target_id != source_id:
                    all_refs.append({
                        'source_id': source_id,
                        'type': 'REFERENCES',
                        'target_id': target_id,
                        'evidence': ev,
                        'source_type': 'EXPLICIT',
                        'confidence': 1.0
                    })

        # ----------------------------------------------------
        # Pattern 3: List Khoản cùng Điều
        # ----------------------------------------------------
        for match in list_khoan_rx.finditer(text):
            full_match_text = match.group(0)
            dieu_match = re.search(r'[Đđ]iều\s+(\d+)', full_match_text)
            if not dieu_match:
                continue
            dieu_num = dieu_match.group(1)
            khoan_part = full_match_text[:dieu_match.start()]
            khoan_nums = re.findall(r'\b\d+\b', khoan_part)
            ev = get_evidence(text, match.start(), match.end())
            
            for k in khoan_nums:
                candidates = [f"{doc_id}_D{dieu_num}_K{k}", f"{doc_id}_D{dieu_num}"]
                target_id = candidates[0] if candidates[0] in node_id_set else candidates[1]
                if target_id != source_id:
                    all_refs.append({
                        'source_id': source_id,
                        'type': 'REFERENCES',
                        'target_id': target_id,
                        'evidence': ev,
                        'source_type': 'EXPLICIT',
                        'confidence': 1.0
                    })

        # ----------------------------------------------------
        # Pattern 4: List Điều (nhiều Điều)
        # ----------------------------------------------------
        for match in list_dieu_rx.finditer(text):
            full_match_text = match.group(0)
            dieu_nums = re.findall(r'\b\d+\b', full_match_text)
            if len(dieu_nums) >= 2:
                ev = get_evidence(text, match.start(), match.end())
                for d in dieu_nums:
                    target_id = f"{doc_id}_D{d}"
                    if target_id != source_id and target_id in node_id_set:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': target_id,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # ----------------------------------------------------
        # Pattern 5: Dẫn chiếu nội bộ "Điều này"
        # ----------------------------------------------------
        if cur_dieu:
            for match in internal_dieu_rx.finditer(text):
                full_m = match.group(0)
                # Lấy số khoản (nếu có)
                k_match = re.findall(r'[Kk]hoản\s+(\d+)|(?:\s*,\s*|\s+và\s+)(\d+)', full_m[:full_m.lower().find('điều')])
                k_nums = [g1 or g2 for g1, g2 in k_match]
                
                # Lấy chữ cái điểm (nếu có)
                diem_part = full_m[:full_m.lower().find('khoản')] if 'khoản' in full_m.lower() else full_m[:full_m.lower().find('điều')]
                d_match = re.findall(r'[Đđ]iểm\s+([a-zđ])|(?:\s*,\s*|\s+và\s+)([a-zđ])\b', diem_part)
                d_letters = [g1 or g2 for g1, g2 in d_match if (g1 or g2).lower() not in ['và', 'của', 'hoặc']]
                
                candidates = []
                if d_letters:
                    for dl in d_letters:
                        if k_nums:
                            for kn in k_nums:
                                candidates.append(f"{doc_id}_D{cur_dieu}_K{kn}_DIEM_{dl.upper()}")
                        else:
                            candidates.append(f"{doc_id}_D{cur_dieu}_DIEM_{dl.upper()}")
                elif k_nums:
                    for kn in k_nums:
                        candidates.append(f"{doc_id}_D{cur_dieu}_K{kn}")
                else:
                    candidates.append(f"{doc_id}_D{cur_dieu}")
                    
                ev = get_evidence(text, match.start(), match.end())
                for cand in candidates:
                    if cand in node_id_set and cand != source_id:
                        all_refs.append({
                            'source_id': source_id,
                            'type': 'REFERENCES',
                            'target_id': cand,
                            'evidence': ev,
                            'source_type': 'EXPLICIT',
                            'confidence': 1.0
                        })

        # ----------------------------------------------------
        # Pattern 6: Single Citation thông thường
        # ----------------------------------------------------
        for match in single_ref_rx.finditer(text):
            diem, khoan, target_dieu = match.groups()
            candidates = []
            if diem and khoan:
                candidates.append(f"{doc_id}_D{target_dieu}_K{khoan}_DIEM_{diem.upper()}")
            if khoan:
                candidates.append(f"{doc_id}_D{target_dieu}_K{khoan}")
            candidates.append(f"{doc_id}_D{target_dieu}")
            
            target_id = None
            for cand in candidates:
                if cand in node_id_set:
                    target_id = cand
                    break
            if not target_id:
                target_id = candidates[-1]
                
            if source_id == target_id:
                continue
                
            ev = get_evidence(text, match.start(), match.end())
            all_refs.append({
                'source_id': source_id,
                'type': 'REFERENCES',
                'target_id': target_id,
                'evidence': ev,
                'source_type': 'EXPLICIT',
                'confidence': 1.0
            })

    # --------------------------------------------------------
    # Khử trùng lặp và gộp bằng chứng (Evidence Merging)
    # --------------------------------------------------------
    unique_refs = {}
    for ref in all_refs:
        key = (ref['source_id'], ref['type'], ref['target_id'])
        if key not in unique_refs:
            unique_refs[key] = ref
        else:
            existing_ev = unique_refs[key]['evidence']
            new_ev = ref['evidence']
            if new_ev not in existing_ev:
                unique_refs[key]['evidence'] = f"{existing_ev} | {new_ev}"
                
    return list(unique_refs.values())

print("Đã tải thành công hàm trích xuất REFERENCES nâng cao (Self-contained, hỗ trợ Range/List/Điều này).")


Đã tải thành công hàm trích xuất REFERENCES nâng cao (Self-contained, hỗ trợ Range/List/Điều này).


## 4. Bộ Kiểm Định Chất Lượng Đồ Thị (Validator V1 — V8)

Kiểm tra 8 điều kiện bắt buộc trước khi lưu file:
- **V1 [DUPLICATE_NODE_ID]:** Mọi `node_id` là duy nhất
- **V2 [ORPHAN_NODE]:** Mọi `parent_id` tồn tại trong nodes
- **V3 [DANGLING_SOURCE]:** `source_id` của mọi quan hệ tồn tại trong nodes
- **V4 [DANGLING_TARGET_STRUCTURAL]:** `target_id` của CONTAINS tồn tại trong nodes
- **V5 [WARNING_DANGLING_REF]:** `target_id` của REFERENCES tồn tại trong nodes (warning)
- **V6 [INVALID_DOCUMENT_ID]:** `document_id` trong mọi node tồn tại trong tập các Document node
- **V7 [INVALID_RELATION_TYPE]:** `type` của quan hệ nằm trong whitelist
- **V8 [MISSING_EVIDENCE]:** REFERENCES phải có `evidence` không rỗng


In [5]:
def validate_graph(nodes, relations):
    """Xác thực toàn bộ các quy tắc kiểm tra tính toàn vẹn của đồ thị (V1 — V8)."""
    errors = []
    warnings = []
    
    node_id_set = {n['node_id'] for n in nodes}
    doc_id_set = {n['node_id'] for n in nodes if n.get('type') == 'Document'}
    
    # V1: Unique node_ids
    node_id_counts = collections.Counter(n['node_id'] for n in nodes)
    for nid, c in node_id_counts.items():
        if c > 1:
            errors.append(f"V1 [DUPLICATE_NODE_ID]: {nid} xuất hiện {c} lần")
            
    # V2: parent_id tồn tại
    for n in nodes:
        pid = n.get('parent_id')
        if pid is not None and pid not in node_id_set:
            errors.append(f"V2 [ORPHAN_NODE]: {n['node_id']} có parent_id không tồn tại: {pid}")
            
    # V3: source_id tồn tại
    for r in relations:
        if r['source_id'] not in node_id_set:
            errors.append(f"V3 [DANGLING_SOURCE]: source_id '{r['source_id']}' không có trong nodes")
            
    # V4: target_id của STRUCTURAL (CONTAINS) tồn tại
    for r in relations:
        if r['type'] == 'CONTAINS' and r['target_id'] not in node_id_set:
            errors.append(f"V4 [DANGLING_TARGET_STRUCTURAL]: CONTAINS target_id '{r['target_id']}' không có trong nodes")
            
    # V5: target_id của REFERENCES tồn tại (cảnh báo nếu liên văn bản chưa nạp)
    for r in relations:
        if r['type'] == 'REFERENCES' and r['target_id'] not in node_id_set:
            warnings.append(f"V5 [WARNING_DANGLING_REF]: REFERENCES target_id '{r['target_id']}' không có trong nodes")
            
    # V6: document_id hợp lệ
    for n in nodes:
        if n['document_id'] not in doc_id_set:
            errors.append(f"V6 [INVALID_DOCUMENT_ID]: node {n['node_id']} có document_id không hợp lệ: {n['document_id']}")
            
    # V7: relation_type trong whitelist
    REL_WHITELIST = {'CONTAINS', 'REFERENCES', 'IMPLEMENTS', 'INTERPRETED_BY', 'AMENDS', 'REPEALS', 'REPLACED_BY', 'RELATED_TO'}
    for r in relations:
        if r['type'] not in REL_WHITELIST:
            errors.append(f"V7 [INVALID_RELATION_TYPE]: relation có type lạ '{r['type']}'")
            
    # V8: evidence không rỗng cho REFERENCES
    for r in relations:
        if r['type'] == 'REFERENCES':
            if not r.get('evidence') or not str(r.get('evidence')).strip():
                errors.append(f"V8 [MISSING_EVIDENCE]: REFERENCES từ {r['source_id']} -> {r['target_id']} thiếu evidence")
                
    return errors, warnings
print("Đã tải bộ kiểm định chất lượng đồ thị.")


Đã tải bộ kiểm định chất lượng đồ thị.


## 5. Chạy Pipeline Xử Lý 


In [6]:
# 1. Đọc dữ liệu thô
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Cấu hình Metadata văn bản pháp lý
DOC_METADATA = {
    'HIEN_PHAP_2013': {
        'document_id': 'HIEN_PHAP_2013',
        'title': 'Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013',
        'number': '15/2013/QH13',
        'type': 'HIEN_PHAP',
        'issuer': 'Quốc hội',
        'effective_from': '2014-01-01',
        'effective_to': None,
        'status': 'CON_HIEU_LUC'
    },
    'BLDS_2015': {
        'document_id': 'BLDS_2015',
        'title': 'Bộ luật Dân sự 2015',
        'number': '91/2015/QH13',
        'type': 'BO_LUAT',
        'issuer': 'Quốc hội',
        'effective_from': '2017-01-01',
        'effective_to': None,
        'status': 'CON_HIEU_LUC'
    }
}

all_nodes = []
all_contains_relations = []

for raw_doc in raw_data:
    meta = raw_doc.get('metadata', {})
    ten_vb = meta.get('ten_van_ban', '').upper()
    if 'HIẾN PHÁP' in ten_vb:
        doc_id = 'HIEN_PHAP_2013'
    elif 'DÂN SỰ' in ten_vb:
        doc_id = 'BLDS_2015'
    else:
        clean_name = re.sub(r'[^a-zA-Z0-9]', '_', ten_vb)[:20]
        doc_id = f"DOC_{clean_name}"
        
    doc_info = DOC_METADATA.get(doc_id, {
        'document_id': doc_id,
        'title': meta.get('ten_van_ban'),
        'number': meta.get('so_hieu'),
        'type': meta.get('loai_van_ban', 'LUAT'),
        'issuer': meta.get('co_quan_ban_hanh'),
        'effective_from': None,
        'effective_to': None,
        'status': 'CON_HIEU_LUC'
    })
    
    # Tạo Document Node với đầy đủ thuộc tính metadata & hiệu lực pháp luật
    all_nodes.append({
        'node_id': doc_id,
        'document_id': doc_id,
        'type': 'Document',
        'label': 'Document',
        'number': doc_info.get('number'),
        'title': doc_info.get('title'),
        'doc_type': doc_info.get('type'),
        'issuer': doc_info.get('issuer'),
        'effective_from': doc_info.get('effective_from'),
        'effective_to': doc_info.get('effective_to'),
        'status': doc_info.get('status'),
        'text': None,
        'parent_id': None
    })
    
    # Duyệt cấu trúc phân cấp bằng thuật toán đệ quy tổng quát
    cau_truc = raw_doc.get('cau_truc', {})
    traverse_tree_recursive(cau_truc, doc_id, doc_id, {}, all_nodes, all_contains_relations)

# Trích xuất các quan hệ REFERENCES (Range, List, Dẫn chiếu nội bộ)
node_id_set = {n['node_id'] for n in all_nodes}
all_reference_relations = extract_references(all_nodes, node_id_set)

# Gộp tất cả quan hệ
all_relations = all_contains_relations + all_reference_relations

# Kiểm định chất lượng toàn vẹn đồ thị
errors, warnings = validate_graph(all_nodes, all_relations)

# Ghi dữ liệu ra 2 file JSON chuẩn cho đồ thị (nodes.json và relations.json)
with open(OUTPUT_DIR / 'nodes.json', 'w', encoding='utf-8') as f:
    json.dump(all_nodes, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / 'relations.json', 'w', encoding='utf-8') as f:
    json.dump(all_relations, f, ensure_ascii=False, indent=2)

print("Đã lưu thành công 2 file đồ thị chuẩn tại: backend/data/processed/ (nodes.json & relations.json)")


Đã lưu thành công 2 file đồ thị chuẩn tại: backend/data/processed/ (nodes.json & relations.json)


## 6. Thống Kê Chi Tiết & Báo Cáo Kiểm Định


In [7]:
print("=" * 68)
print("             BẢNG THỐNG KÊ CHI TIẾT NODES ĐỒ THỊ")
print("=" * 68)
doc_count = sum(1 for n in all_nodes if n.get('type') == 'Document')
print(f"Tổng số văn bản (Documents): {doc_count:,}")
print(f"Tổng số nodes toàn đồ thị  : {len(all_nodes):,}")
node_types = collections.Counter(n['type'] for n in all_nodes)
for t, count in node_types.most_common():
    print(f"  - {t:<15}: {count:,}")

print("\n" + "=" * 68)
print("           BẢNG THỐNG KÊ QUAN HỆ (RELATIONSHIPS)")
print("=" * 68)
print(f"Tổng số quan hệ: {len(all_relations):,}")
rel_types = collections.Counter(r['type'] for r in all_relations)
for t, count in rel_types.most_common():
    print(f"  - {t:<15}: {count:,}")

print("\n" + "=" * 68)
print("           KẾT QUẢ KIỂM ĐỊNH TÍNH TOÀN VẸN (V1 — V8)")
print("=" * 68)
if errors:
    print(f"[THẤT BẠI] Phát hiện {len(errors)} lỗi nghiêm trọng:")
    for e in errors:
        print(f"  x {e}")
else:
    print(">> ĐẠT CHUẨN 100%! Không có orphan node hay ID bị trùng.")

if warnings:
    print(f"\n[CẢNH BÁO] {len(warnings)} cảnh báo:")
    for w in warnings[:5]:
        print(f"  ! {w}")
else:
    print(">> 0 Cảnh báo! 100% target của REFERENCES tồn tại trong cơ sở dữ liệu.")


             BẢNG THỐNG KÊ CHI TIẾT NODES ĐỒ THỊ
Tổng số văn bản (Documents): 2
Tổng số nodes toàn đồ thị  : 2,891
  - Clause         : 1,662
  - Article        : 809
  - Point          : 312
  - Section        : 39
  - Chapter        : 38
  - SubSection     : 23
  - Part           : 6
  - Document       : 2

           BẢNG THỐNG KÊ QUAN HỆ (RELATIONSHIPS)
Tổng số quan hệ: 3,189
  - CONTAINS       : 2,889
  - REFERENCES     : 300

           KẾT QUẢ KIỂM ĐỊNH TÍNH TOÀN VẸN (V1 — V8)
>> ĐẠT CHUẨN 100%! Không có orphan node hay ID bị trùng.
>> 0 Cảnh báo! 100% target của REFERENCES tồn tại trong cơ sở dữ liệu.


## 7. Kiểm Thử Khả Năng Thích Ứng Cấu Trúc Dị Biệt (Heterogeneous Structure Test)

Chứng minh thuật toán đệ quy chạy ổn định trên các cấu trúc phân cấp phi chuẩn:


In [8]:
# 1. Cấu trúc đầy đủ: Chương -> Mục -> Điều -> Khoản -> Điểm (Nghị quyết A)
mock_nq_a = {
    'loai': 'chuong',
    'so': 'I',
    'ten': 'CHƯƠNG THỬ NGHIỆM',
    'cac_muc': [{
        'loai': 'muc',
        'so': 1,
        'ten': 'Mục 1 Thử Nghiệm',
        'cac_dieu': [{
            'loai': 'dieu',
            'so': 1,
            'tieu_de': 'Điều 1 Thử Nghiệm',
            'cac_khoan': [{
                'loai': 'khoan',
                'so': 1,
                'noi_dung': 'Khoản 1',
                'cac_diem': [{'loai': 'diem', 'so': 'a', 'noi_dung': 'Điểm a'}]
            }]
        }]
    }]
}

# 2. Cấu trúc rút gọn: Điều -> Khoản -> Điểm trực tiếp (Nghị quyết B)
mock_nq_b = {
    'loai': 'dieu',
    'so': 10,
    'tieu_de': 'Điều 10 Rút Gọn',
    'cac_khoan': [{
        'loai': 'khoan',
        'so': 1,
        'noi_dung': 'Nội dung khoản 1',
        'cac_diem': [{'loai': 'diem', 'so': 'b', 'noi_dung': 'Điểm b'}]
    }]
}

test_nodes, test_rels = [], []
traverse_tree_recursive(mock_nq_a, 'TEST_NQ_A', 'TEST_NQ_A', {}, test_nodes, test_rels)
traverse_tree_recursive(mock_nq_b, 'TEST_NQ_B', 'TEST_NQ_B', {}, test_nodes, test_rels)

print(f"Thử nghiệm cấu trúc dị biệt thành công! Sinh {len(test_nodes)} nodes và {len(test_rels)} quan hệ CONTAINS.")
for n in test_nodes:
    print(f"  Node: {n['node_id']:<35} (Type: {n['type']}, Parent: {n['parent_id']})")


Thử nghiệm cấu trúc dị biệt thành công! Sinh 8 nodes và 8 quan hệ CONTAINS.
  Node: TEST_NQ_A_CH1                       (Type: Chapter, Parent: TEST_NQ_A)
  Node: TEST_NQ_A_CH1_M1                    (Type: Section, Parent: TEST_NQ_A_CH1)
  Node: TEST_NQ_A_D1                        (Type: Article, Parent: TEST_NQ_A_CH1_M1)
  Node: TEST_NQ_A_D1_K1                     (Type: Clause, Parent: TEST_NQ_A_D1)
  Node: TEST_NQ_A_D1_K1_DIEM_A              (Type: Point, Parent: TEST_NQ_A_D1_K1)
  Node: TEST_NQ_B_D10                       (Type: Article, Parent: TEST_NQ_B)
  Node: TEST_NQ_B_D10_K1                    (Type: Clause, Parent: TEST_NQ_B_D10)
  Node: TEST_NQ_B_D10_K1_DIEM_B             (Type: Point, Parent: TEST_NQ_B_D10_K1)
